# File 3: Tích Hợp Google Gemini API
Dự đoán bệnh răng miệng từ ảnh X-quang và đưa ra lời khuyên bằng Gemini.

## 1. Cài đặt & Import thư viện

In [ ]:
# Cài đặt thư viện nếu chưa có
# !pip install google-generativeai


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import google.generativeai as genai
import base64, textwrap, os



## 2. Cấu hình

In [3]:
IMG_SIZE    = (128, 128)
class_names = ['Cavity', 'Fillings', 'Impacted Tooth', 'Implant', 'Normal']

# Load model đã huấn luyện
model = load_model('../Model/best_model.keras')
print("Model loaded:", model.input_shape)


Model loaded: (None, 128, 128, 3)


## 3. Cấu hình Gemini API

In [ ]:
user_secrets = UserSecretsClient()
api_key      = user_secrets.get_secret("API")

genai.configure(api_key=api_key)
model_AI = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini API configured.")


## 4. Hàm dự đoán bệnh

In [ ]:
def predict_disease(img_path, model, class_names):
    """Load ảnh, predict và trả về tên bệnh + confidence."""    img       = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred_probs = model.predict(img_array, verbose=0)
    pred_class = np.argmax(pred_probs)
    return class_names[pred_class], float(np.max(pred_probs))

def show_prediction(img_path, disease, confidence):
    """Hiển thị ảnh cùng kết quả dự đoán."""    img = image.load_img(img_path, target_size=(256, 256))
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Result: {disease}\nConfidence: {confidence:.2%}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()


## 5. Hàm đưa ra lời khuyên bằng Gemini

In [ ]:
def generate_advice(disease):
    """Gọi Gemini để tạo lời khuyên nha khoa."""    
    if disease != "Normal":
        prompt = f"""
Tôi có vấn đề về răng miệng: {disease}
Bạn là một bác sĩ nha khoa.
Hãy cho tôi 3 lời khuyên ngắn gọn để khắc phục tình trạng đó.
Yêu cầu định dạng:
- Đánh số thứ tự 1. 2. 3.
- Không dùng ký tự * hoặc ** hoặc markdown bất kỳ
- Chỉ dùng văn bản thuần túy
"""
    else:
        prompt = """
Bạn là một bác sĩ nha khoa.
Hãy cho tôi 3 lời khuyên ngắn gọn để giữ răng khỏe đẹp.
Yêu cầu định dạng:
- Đánh số thứ tự 1. 2. 3.
- Không dùng ký tự * hoặc ** hoặc markdown bất kỳ
- Chỉ dùng văn bản thuần túy
"""
    response = model_AI.generate_content(prompt)
    return response.text


## 6. Hàm phân tích ảnh bị phân loại sai bằng Gemini Vision

In [ ]:
def encode_image_to_base64(img_path):
    with open(img_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def analyze_misclassified_image(img_path, true_class, pred_class):
    """Gọi Gemini Vision để phân tích lý do model nhầm."""    img_b64 = encode_image_to_base64(img_path)
    prompt  = f"""
Đây là ảnh X-quang răng bị phân loại sai bởi mô hình AI.
- Nhãn thật: {true_class}
- Mô hình dự đoán nhầm thành: {pred_class}

Với tư cách là bác sĩ nha khoa và chuyên gia AI, hãy phân tích ngắn gọn:
1. Đặc điểm hình ảnh nào khiến mô hình nhầm lẫn?
2. Điểm khác biệt giữa {true_class} và {pred_class} trong ảnh X-quang?

Trả lời ngắn gọn, không dùng markdown hay ký tự đặc biệt.
"""
    response = model_AI.generate_content([
        {"mime_type": "image/png", "data": img_b64},
        prompt
    ])
    return response.text


## 7. Demo: Dự đoán + Lời khuyên cho một ảnh

In [ ]:
# Thay đường dẫn ảnh tùy ý
img_path = '/kaggle/input/datasets/trngvnthin/dataset/DataSet/test/Implant/0053_jpg.rf.0a39e4969cb730a0e08d415d819e14d9_segment_28.png'

disease, confidence = predict_disease(img_path, model, class_names)
show_prediction(img_path, disease, confidence)

print(f"\nBệnh dự đoán: {disease}  (confidence: {confidence:.2%})")
print("\nLời khuyên từ Gemini:")
print(generate_advice(disease))

if confidence < 0.8:
    print("\n Chú ý: Confidence thấp, hãy đến nha khoa để kiểm tra lại.")


## 8. Phân tích toàn bộ ảnh bị phân loại sai bằng Gemini Vision

In [ ]:
misclassified_dir = "/kaggle/working/misclassified"
MAX_PER_CLASS = 1   # Tăng lên nếu cần (tốn API quota)

print("PHÂN TÍCH LÝ DO PHÂN LOẠI SAI\n")

for true_class in sorted(os.listdir(misclassified_dir)):
    class_path = os.path.join(misclassified_dir, true_class)
    if not os.path.isdir(class_path):
        continue

    files = os.listdir(class_path)
    print(f"{'='*50}")
    print(f"Class: {true_class}  ({len(files)} ảnh sai)")
    print(f"{'='*50}")

    for fname in files[:MAX_PER_CLASS]:
        img_path   = os.path.join(class_path, fname)
        pred_class = fname.split("pred_")[1].split("__")[0]

        # Hiển thị ảnh
        img = mpimg.imread(img_path)
        plt.figure(figsize=(3, 3))
        plt.imshow(img)
        plt.title(f"True: {true_class} | Pred: {pred_class}", color='red', fontsize=9)
        plt.axis('off'); plt.show()

        # Phân tích bằng Gemini
        print(f"Ảnh: {fname}")
        print(f"Nhầm: {true_class} → {pred_class}")
        analysis = analyze_misclassified_image(img_path, true_class, pred_class)
        print(f"Phân tích:\n{textwrap.fill(analysis, width=80)}")
        print()
